# Knowledge Distillation: Llama-3.1-8B → Llama-3.2-1B
**CS 455 Term Project** — Ali Kumral, Revna Demirkale

---
## ⚠️ MUST RUN FIRST — run the cell below after every runtime restart
Everything else in this notebook depends on it. It is safe to re-run at any time.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# MUST RUN FIRST — run after every runtime restart before anything else
# ═══════════════════════════════════════════════════════════════════════
import os, sys

REPO_URL   = "https://github.com/alikumral/Knowledge-Distillation-Transferring-Mathematical-Reasoning"
REPO_NAME  = "Knowledge-Distillation-Transferring-Mathematical-Reasoning"
REPO_PATH  = f"/content/{REPO_NAME}"
DRIVE_HOME = "/content/drive/MyDrive/CS455/TermProject/Knowledge-Distillation-Transferring-Mathematical-Reasoning"

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

if not os.path.exists(REPO_PATH):
    print("Cloning repo...")
    os.system(f"git clone {REPO_URL}")
else:
    print("Repo exists, pulling latest...")
    os.system(f"git -C {REPO_PATH} pull")
    # Clear stale module cache so pulled changes take effect in this kernel.
    # (git pull updates files on disk; without this, Python keeps using the
    #  old in-memory version of every mathdistill module.)
    for _k in list(sys.modules.keys()):
        if "mathdistill" in _k:
            del sys.modules[_k]
    print("Module cache cleared.")

os.chdir(REPO_PATH)
if os.path.join(REPO_PATH, "src") not in sys.path:
    sys.path.insert(0, os.path.join(REPO_PATH, "src"))

os.system("pip install -q -e . -r requirements.txt")

from google.colab import drive
drive.mount("/content/drive", force_remount=False)
os.environ["MATHDISTILL_HOME"] = DRIVE_HOME
os.makedirs(DRIVE_HOME, exist_ok=True)

from google.colab import userdata
from huggingface_hub import login
login(token=userdata.get("HF_TOKEN"), add_to_git_credential=False)

import torch
gpu  = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "⚠ NO GPU"
vram = f"{torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB" if torch.cuda.is_available() else ""
print(f"\n{'='*55}")
print(f"  GPU:       {gpu}  {vram}")
print(f"  Repo:      {os.getcwd()}")
print(f"  Artifacts: {os.environ['MATHDISTILL_HOME']}")
print(f"{'='*55}")
print("  Setup complete. Ready to run any stage below.")
print(f"{'='*55}")

---
## [OPTIONAL] Smoke test — already passed, skip this
⚠️ **Do NOT run before Stage 1.** Loading both models dirties VRAM even after `del model`.

In [ ]:
# OPTIONAL — already passed on 2026-05-29. Skip before Stage 1.
import torch
from mathdistill.utils import load_config
from mathdistill.models import load_model_4bit, load_tokenizer, render_prompt
from mathdistill.prompts import build_messages
from mathdistill.answers import extract_pred_number

tcfg = load_config("configs/teacher_gen.yaml")
q = ("Natalia sold clips to 48 friends in April, and half as many in May. "
     "How many clips did she sell altogether?")

for model_id in [tcfg["model_id"], "meta-llama/Llama-3.2-1B-Instruct"]:
    print("=" * 60, "\nLoading", model_id)
    tok = load_tokenizer(model_id)
    model = load_model_4bit(model_id, tcfg["quantization"])
    enc = tok(render_prompt(tok, build_messages(q)), return_tensors="pt").to(model.device)
    out = model.generate(**enc, max_new_tokens=256, do_sample=False, pad_token_id=tok.pad_token_id)
    text = tok.decode(out[0, enc["input_ids"].shape[1]:], skip_special_tokens=True)
    print(text)
    print(">> extracted:", extract_pred_number(text), "(expected 72)")
    del model
    torch.cuda.empty_cache()

---
## Stage 1 — Teacher CoT Generation

**Strategy:** 3 passes × 1 sample/problem × batch_size=16 ≈ **14h per pass** (vs. 110h for the original k=4, batch_size=4 design).

| Pass | `PASS =` | sample_idx written | Expected time | Status |
|---|---|---|---|---|
| 0 | `0` | 0 for all 7,473 problems | ~14h | ✓ pilot done (30 problems) |
| 1 | `1` | 1 for all 7,473 problems | ~14h | pending |
| 2 | `2` | 2 for all 7,473 problems | ~14h | pending |

**Each pass is independent and resumable** — if Colab disconnects, re-run the MUST-RUN-FIRST cell and this cell with the same `PASS` value. It picks up where it left off.

After all 3 passes: run Stage 2 to build SFT datasets.

In [ ]:
from mathdistill.generate import generate_teacher_traces
from mathdistill.utils import load_config

PILOT = False  # True = 30 problems (~3 min to validate). False = full 7,473 problems.
PASS  = 0      # ← change to 0, 1, or 2 for each separate generation pass

cfg = load_config("configs/teacher_gen.yaml")
cfg["pass_id"] = PASS
if PILOT:
    cfg["limit"] = 30

print(f"Starting pass {PASS} | pilot={PILOT} | batch_size={cfg['batch_size']} | k={cfg['generation']['k_samples']}")
generate_teacher_traces(cfg, load_config("configs/data.yaml"))

---
## Stage 2 — Rejection Sampling & Build SFT Datasets

**Run after each pass** to check acceptance stats (Risk-1 check).  
**Run `build_all_sft_datasets`** only after all 3 passes are complete.

Expected after 3 passes: ~6,700 problems covered (90% of 7,473), ~20k total traces.

In [ ]:
import json
from mathdistill.reject import rejection_sample, acceptance_stats, build_all_sft_datasets
from mathdistill.utils import artifact_path, load_config

traces_path   = artifact_path("traces", "teacher_traces.jsonl")
accepted_path = artifact_path("sft", "accepted.jsonl")

rejection_sample(traces_path, accepted_path)
stats = acceptance_stats(accepted_path)
print(json.dumps(stats, indent=2))

if stats["n_problems_with_ge1"] < 18:  # <60% of 30-problem pilot
    print("\n⚠ Low acceptance — consider Plan B (more samples or swap teacher)")
else:
    print("\n✓ Acceptance looks good")

# ── Uncomment ONLY after all 3 passes are complete ─────────────────────
# build_all_sft_datasets(accepted_path, load_config("configs/data.yaml"))
# print("SFT datasets written to Drive.")

---
## Stage 3 — QLoRA Fine-tuning
Change `CONFIG` and `SEED` per run (~2h each). Spread across sessions.

In [ ]:
# Clear any stale mathdistill imports before loading (safe to run multiple times)
import sys
for _k in list(sys.modules.keys()):
    if "mathdistill" in _k:
        del sys.modules[_k]

from mathdistill.train import train_one
from mathdistill.utils import load_config

CONFIG = "configs/train_teacher_1.yaml"   # train_teacher_1 | train_teacher_3 | train_gold
SEED   = 0                                # 0, 1, 2

train_one(load_config(CONFIG), load_config("configs/data.yaml"), seed=SEED)

---
## Stage 4 — Evaluation
Greedy decoding on GSM8K test + MATH-200. Accuracy with 95% bootstrap CIs.

In [ ]:
import sys
for _k in list(sys.modules.keys()):
    if "mathdistill" in _k:
        del sys.modules[_k]

STUDENT = "meta-llama/Llama-3.2-1B-Instruct"
TEACHER = "meta-llama/Llama-3.1-8B-Instruct"

# Each run_eval.py call is a separate subprocess — model memory is freed between conditions.
# Run them one at a time; comment out conditions you haven't trained yet.

# (i) zero-shot 1B — no adapter
!python scripts/run_eval.py --model-id {STUDENT} --name zeroshot

# (ii) gold CoT, seed 0
!python scripts/run_eval.py --model-id {STUDENT} --adapter adapters/gold/seed0 --name gold_seed0

# (iii) teacher_1, seed 0
!python scripts/run_eval.py --model-id {STUDENT} --adapter adapters/teacher_1/seed0 --name teacher_1_seed0

# (iv) teacher_3, seed 0
!python scripts/run_eval.py --model-id {STUDENT} --adapter adapters/teacher_3/seed0 --name teacher_3_seed0

# (v) 8B teacher upper bound — takes longer (~3x eval time)
!python scripts/run_eval.py --model-id {TEACHER} --name teacher

In [ ]:
import glob, os, pandas as pd
from mathdistill.utils import read_jsonl
from mathdistill.metrics import accuracy, bootstrap_ci

pred_dir = os.path.join(os.environ["MATHDISTILL_HOME"], "results", "predictions")
rows = []
for path in sorted(glob.glob(os.path.join(pred_dir, "*.jsonl"))):
    data = read_jsonl(path)
    correct = [r["correct"] for r in data]
    lo, hi = bootstrap_ci(correct)
    rows.append({"condition": os.path.basename(path).replace(".jsonl", ""),
                 "n": len(correct), "acc": round(accuracy(correct), 4),
                 "ci_lo": round(lo, 4), "ci_hi": round(hi, 4)})
pd.DataFrame(rows)

---
## Analysis 1 — Per-difficulty Error Analysis
## Analysis 2 — Generation-length Distribution
## Analysis 3 — Qualitative Trace Study

Run the cells below in order. No GPU needed — all work on the saved prediction files.

**Step 1:** Load all predictions + GSM8K test metadata (run first, others depend on it).

In [ ]:
# ── Step 1: Load all predictions and GSM8K test metadata ─────────────────────
import os, numpy as np, pandas as pd
from datasets import load_dataset
from mathdistill.utils import read_jsonl

PRED = os.path.join(os.environ["MATHDISTILL_HOME"], "results", "predictions")
FIG  = os.path.join(os.environ["MATHDISTILL_HOME"], "results", "figures")
os.makedirs(FIG, exist_ok=True)

# Load GSM8K test split (needed for questions, gold answers, difficulty buckets)
test_data = list(load_dataset("openai/gsm8k", "main")["test"])
N = len(test_data)  # 1319

# Gold CoT length (word count of solution text before ####) → proxy for difficulty
gold_cot_lengths = [
    len(row["answer"].split("####")[0].strip().split())
    for row in test_data
]
sorted_len = sorted(gold_cot_lengths)
t1, t2 = sorted_len[N // 3], sorted_len[2 * N // 3]

def difficulty(length):
    if length <= t1:   return "easy"
    elif length <= t2: return "medium"
    else:              return "hard"

diff_labels = [difficulty(l) for l in gold_cot_lengths]
print(f"Difficulty thresholds (by gold CoT word count):")
print(f"  easy   : ≤ {t1} words")
print(f"  medium : {t1+1}–{t2} words")
print(f"  hard   : > {t2} words")
print(f"  counts : {diff_labels.count('easy')} / {diff_labels.count('medium')} / {diff_labels.count('hard')}")

# Load all 5 GSM8K prediction files into a dict keyed by condition name
# Note: teacher was saved as "teache" (typo in --name flag, one letter short)
CONDS = {
    "zero_shot":  "zeroshot_gsm8k.jsonl",
    "gold":       "gold_seed0_gsm8k.jsonl",
    "teacher_1":  "teacher_1_seed0_gsm8k.jsonl",
    "teacher_3":  "teacher_3_seed0_gsm8k.jsonl",
    "teacher_8B": "teache_gsm8k.jsonl",
}

preds = {}
for name, fname in CONDS.items():
    path = os.path.join(PRED, fname)
    rows = read_jsonl(path)
    preds[name] = {r["problem_id"]: r for r in rows}
    print(f"  {name}: {len(rows)} predictions loaded")

print("\nData loaded — ready for analysis cells below.")

### Analysis 1 — Per-difficulty Error Analysis (Step 2)
Accuracy bucketed by gold-CoT length tertiles (easy / medium / hard).
Shows where the student gains most from distillation and where it still trails the teacher.

In [ ]:
import matplotlib.pyplot as plt

# ── Per-difficulty accuracy table ─────────────────────────────────────────────
rows = []
for cond_name in CONDS:
    cond_preds = preds[cond_name]
    buckets = {"easy": [], "medium": [], "hard": [], "overall": []}
    for pid in range(N):
        if pid not in cond_preds:
            continue
        correct = cond_preds[pid]["correct"]
        buckets[diff_labels[pid]].append(correct)
        buckets["overall"].append(correct)
    row = {"condition": cond_name}
    for b in ["easy", "medium", "hard", "overall"]:
        vals = buckets[b]
        row[b] = round(sum(vals) / len(vals), 4) if vals else float("nan")
    rows.append(row)

df_diff = pd.DataFrame(rows)
print("── Accuracy by difficulty bucket ──")
print(df_diff.to_string(index=False))

# ── Figure F1: grouped bar chart ──────────────────────────────────────────────
x = np.arange(3)
buckets_plot = ["easy", "medium", "hard"]
width = 0.15
fig, ax = plt.subplots(figsize=(10, 5))

colors = ["#aec7e8", "#ffbb78", "#98df8a", "#ff9896", "#c5b0d5"]
for i, (_, row) in enumerate(df_diff.iterrows()):
    vals = [row[b] for b in buckets_plot]
    ax.bar(x + i * width, vals, width, label=row["condition"], color=colors[i])

ax.set_xticks(x + 2 * width)
ax.set_xticklabels(["Easy\n(≤%d words)" % t1,
                    "Medium\n(%d-%d words)" % (t1+1, t2),
                    "Hard\n(>%d words)" % t2])
ax.set_ylabel("Accuracy")
ax.set_title("GSM8K accuracy by problem difficulty (gold-CoT length tertiles)")
ax.legend(fontsize=8)
ax.set_ylim(0, 1)
ax.axhline(0, color="black", linewidth=0.5)
plt.tight_layout()
plt.savefig(os.path.join(FIG, "F1_difficulty_accuracy.png"), dpi=150, bbox_inches="tight")
plt.show()
print("Saved -> F1_difficulty_accuracy.png")

### Analysis 2 — Generation-length Distribution (Step 3)
Reveals verbosity inheritance: distilled models learned to generate longer reasoning chains like the teacher.
This explains the 2× latency increase (teacher_1 vs zero-shot).

In [ ]:
# ── Generation-length statistics ──────────────────────────────────────────────
len_rows = []
all_lengths = {}
for cond_name in CONDS:
    lengths = [r["gen_tokens"] for r in preds[cond_name].values()]
    all_lengths[cond_name] = lengths
    len_rows.append({
        "condition": cond_name,
        "mean":   round(float(np.mean(lengths)), 1),
        "median": round(float(np.median(lengths)), 1),
        "p25":    round(float(np.percentile(lengths, 25)), 1),
        "p75":    round(float(np.percentile(lengths, 75)), 1),
        "p90":    round(float(np.percentile(lengths, 90)), 1),
        "max":    int(np.max(lengths)),
    })

df_len = pd.DataFrame(len_rows)
print("── Generation length statistics (tokens) ──")
print(df_len.to_string(index=False))

# ── Figure F2: box plots + overlaid histograms ────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Box plot
data_plot = [all_lengths[c] for c in CONDS]
bp = ax1.boxplot(data_plot, patch_artist=True, notch=False)
for patch, color in zip(bp["boxes"], colors):
    patch.set_facecolor(color)
ax1.set_xticklabels(list(CONDS.keys()), rotation=20, ha="right")
ax1.set_ylabel("Generated tokens")
ax1.set_title("Generation length distribution (GSM8K test)")

# Histogram overlay for zero_shot vs teacher_1 vs teacher_8B (most interesting trio)
for cond, color, label in [
    ("zero_shot",  "#aec7e8", "zero-shot 1B"),
    ("teacher_1",  "#98df8a", "teacher_1 (distilled)"),
    ("teacher_8B", "#c5b0d5", "8B teacher"),
]:
    ax2.hist(all_lengths[cond], bins=40, alpha=0.55, color=color,
             label=label, density=True)
ax2.set_xlabel("Generated tokens")
ax2.set_ylabel("Density")
ax2.set_title("Verbosity: zero-shot vs distilled vs teacher")
ax2.legend()

plt.tight_layout()
plt.savefig(os.path.join(FIG, "F2_generation_lengths.png"), dpi=150, bbox_inches="tight")
plt.show()
print("Saved -> F2_generation_lengths.png")

### Analysis 3 — Qualitative Trace Study (Step 4)
Side-by-side comparison of zero-shot vs teacher_1 on matched problems.
Picks: 2 × teacher_1 wins, 1 × teacher_1 loss, 1 × both correct (style comparison).

In [ ]:
def show_case(pid, label, max_chars=600):
    """Print a formatted comparison case for the report."""
    q   = test_data[pid]["question"]
    ans = test_data[pid]["answer"].split("####")[-1].strip()
    zs  = preds["zero_shot"][pid]
    t1p = preds["teacher_1"][pid]
    sep = "─" * 70
    print(f"\n{'═'*70}")
    print(f"  [{label}]  Problem ID {pid}  |  difficulty: {diff_labels[pid]}")
    print(f"{'═'*70}")
    print(f"QUESTION:  {q}")
    print(f"GOLD ANSWER: {ans}")
    print(sep)
    zs_tag  = "✓" if zs["correct"]  else "✗"
    t1_tag  = "✓" if t1p["correct"] else "✗"
    print(f"ZERO-SHOT {zs_tag} ({zs['gen_tokens']} tokens):\n{zs['completion'][:max_chars]}")
    print(sep)
    print(f"TEACHER_1 {t1_tag} ({t1p['gen_tokens']} tokens):\n{t1p['completion'][:max_chars]}")

# ── Find cases ────────────────────────────────────────────────────────────────
t1_wins  = [pid for pid in range(N) if pid in preds["teacher_1"] and pid in preds["zero_shot"]
             and preds["teacher_1"][pid]["correct"] and not preds["zero_shot"][pid]["correct"]]
t1_losses = [pid for pid in range(N) if pid in preds["teacher_1"] and pid in preds["zero_shot"]
              and not preds["teacher_1"][pid]["correct"] and preds["zero_shot"][pid]["correct"]]
both_correct = [pid for pid in range(N) if pid in preds["teacher_1"] and pid in preds["zero_shot"]
                and preds["teacher_1"][pid]["correct"] and preds["zero_shot"][pid]["correct"]]

print(f"teacher_1 wins  (t1✓ zs✗): {len(t1_wins)}")
print(f"teacher_1 losses (t1✗ zs✓): {len(t1_losses)}")
print(f"Both correct:                {len(both_correct)}")
print(f"Both wrong:                  {N - len(t1_wins) - len(t1_losses) - len(both_correct)}")

# ── Show 2 wins, 1 loss, 1 style comparison ───────────────────────────────────
# Win 1: a medium-difficulty win to show the teacher's CoT style helps
medium_wins = [p for p in t1_wins if diff_labels[p] == "medium"]
show_case(medium_wins[0], "WIN — teacher_1 correct, zero-shot wrong (medium)")

# Win 2: a hard win
hard_wins = [p for p in t1_wins if diff_labels[p] == "hard"]
if hard_wins:
    show_case(hard_wins[0], "WIN — teacher_1 correct, zero-shot wrong (hard)")

# Loss: teacher_1 wrong despite verbosity
if t1_losses:
    show_case(t1_losses[0], "LOSS — zero-shot correct, teacher_1 wrong")

# Style comparison: both correct but different verbosity
if both_correct:
    show_case(both_correct[0], "STYLE — both correct, compare reasoning length")

---
# Tier-1 Rigor (Essential)

**Run now (no GPU):** Analysis 4 (significance tests) + Analysis 5 (MATH grader sanity check).
**Then (GPU):** Seeds 1 & 2 for teacher_1 and teacher_3, then Analysis 6 (multi-seed aggregation).

All cells below depend on the **Step 1 data-loading cell** having been run.

In [ ]:
# ── Analysis 4: McNemar exact paired significance tests (no GPU) ──────────────
# Paired test on the same 1,319 GSM8K items. Tests whether accuracy differences
# between conditions are statistically significant given the per-item pairing.
from scipy.stats import binomtest

# correctness dict per condition: {problem_id: bool}
corr = {name: {pid: r["correct"] for pid, r in preds[name].items()} for name in CONDS}

def mcnemar_exact(a_corr, b_corr):
    """Exact (binomial) McNemar test on discordant pairs."""
    pids = [p for p in a_corr if p in b_corr]
    b = sum(1 for p in pids if a_corr[p] and not b_corr[p])   # A right, B wrong
    c = sum(1 for p in pids if not a_corr[p] and b_corr[p])   # A wrong, B right
    n = b + c
    if n == 0:
        return b, c, 1.0
    p = binomtest(min(b, c), n, 0.5, alternative="two-sided").pvalue
    return b, c, p

print(f"{'comparison':<28} {'A-only':>7} {'B-only':>7} {'p-value':>12}  hypothesis")
print("-" * 75)
tests = [
    ("teacher_1", "zero_shot", "H1: distillation works"),
    ("teacher_1", "gold",      "H2: teacher > gold"),
    ("teacher_1", "teacher_3", "H3: more traces help"),
    ("gold",      "zero_shot", "gold vs zero-shot"),
]
for A, B, hyp in tests:
    b, c, p = mcnemar_exact(corr[A], corr[B])
    sig = "***" if p < 0.001 else ("**" if p < 0.01 else ("*" if p < 0.05 else "n.s."))
    print(f"{A+' vs '+B:<28} {b:>7} {c:>7} {p:>12.2e}  {sig:<5} {hyp}")
print("\n(A-only = problems A got right but B got wrong, and vice versa)")

In [ ]:
# ── Analysis 5: MATH grader sanity check (no GPU) ────────────────────────────
# Our MATH matcher is approximate (string/numeric, no CAS). Before trusting the
# "no OOD generalization" (H4) finding, inspect cases marked WRONG to confirm the
# grader isn't unfairly rejecting correct answers in a different format.
from mathdistill.utils import read_jsonl

math_wrong = [r for r in read_jsonl(os.path.join(PRED, "teacher_1_seed0_math.jsonl"))
              if not r["correct"]]
print(f"teacher_1 MATH: {len(math_wrong)} marked incorrect. Inspecting first 15 for grader fairness:\n")

suspect = 0
for r in math_wrong[:15]:
    tail = r["completion"][-220:].replace("\n", " ")
    print("=" * 72)
    print(f"GOLD: {r['gold']!r}   |   EXTRACTED PRED: {r['pred']!r}")
    print(f"completion tail: ...{tail}")

print("\n" + "=" * 72)
print("MANUAL CHECK: For each case above, is the model's actual final answer")
print("equal to GOLD but the grader missed it (formatting)? Count those.")
print("If <2-3 of 15 are grader errors -> the H4 degradation is REAL.")
print("If many are grader errors -> note MATH numbers are a lower bound.")

### Seeds 1 & 2 for teacher_1 and teacher_3 (GPU)

Trains 4 adapters (teacher_1 seed1/seed2, teacher_3 seed1/seed2) then evaluates them on GSM8K only.
Use A100 if you have units (fast, ~2.5h total) or free T4 (slower). **Training does not auto-resume mid-run** — if it disconnects, the completed adapters are saved; just edit the loop to skip them.

Then run **Analysis 6** to get mean ± std across the 3 seeds.

In [ ]:
# ── Train seeds 1 & 2 for teacher_1 and teacher_3 ────────────────────────────
import sys
for _k in list(sys.modules.keys()):
    if "mathdistill" in _k:
        del sys.modules[_k]

from mathdistill.train import train_one
from mathdistill.utils import load_config

dcfg = load_config("configs/data.yaml")
for config_path in ["configs/train_teacher_1.yaml", "configs/train_teacher_3.yaml"]:
    cfg = load_config(config_path)
    for seed in [1, 2]:
        print(f"\n{'#'*60}\n# Training {cfg['condition']} seed {seed}\n{'#'*60}")
        train_one(cfg, dcfg, seed=seed)
print("\nAll seed adapters trained.")

In [ ]:
# ── Evaluate the new seed adapters on GSM8K + MATH (resumable: skips done ones) ─
import os, subprocess
STUDENT = "meta-llama/Llama-3.2-1B-Instruct"
PRED_DIR = os.path.join(os.environ["MATHDISTILL_HOME"], "results", "predictions")

for cond in ["teacher_1", "teacher_3"]:
    for seed in [1, 2]:
        name = f"{cond}_seed{seed}"
        # A condition is "done" only if BOTH benchmark files exist (eval writes at the end).
        done = (os.path.exists(os.path.join(PRED_DIR, f"{name}_gsm8k.jsonl"))
                and os.path.exists(os.path.join(PRED_DIR, f"{name}_math.jsonl")))
        if done:
            print(f"SKIP {name} — already evaluated")
            continue
        print(f"\n{'#'*60}\n# Evaluating {name} (GSM8K + MATH)\n{'#'*60}")
        subprocess.run([
            "python", "scripts/run_eval.py",
            "--config", "configs/eval_seeds.yaml",
            "--model-id", STUDENT,
            "--adapter", f"adapters/{cond}/seed{seed}",
            "--name", name,
        ], check=True)
print("\nAll seed evals done. Run Analysis 6 below.")

In [ ]:
# ── Analysis 6: multi-seed aggregation (mean ± std) for GSM8K and MATH ────────
import numpy as np
from mathdistill.utils import read_jsonl
from mathdistill.metrics import accuracy

def acc_of(fname):
    path = os.path.join(PRED, fname)
    return accuracy([r["correct"] for r in read_jsonl(path)]) if os.path.exists(path) else None

def seed_accs(cond, bench):
    vals = [acc_of(f"{cond}_seed{s}_{bench}.jsonl") for s in [0, 1, 2]]
    return [v for v in vals if v is not None]

def fmt(accs):
    return (round(float(np.mean(accs)), 4), round(float(np.std(accs)), 4), len(accs))

rows = []
# deterministic baselines (single run)
for cond, gf, mf in [("zero_shot", "zeroshot_gsm8k.jsonl", "zeroshot_math.jsonl"),
                     ("gold", "gold_seed0_gsm8k.jsonl", "gold_seed0_math.jsonl"),
                     ("teacher_8B", "teache_gsm8k.jsonl", "teache_math.jsonl")]:
    rows.append({"condition": cond, "n_seeds": 1,
                 "gsm8k_mean": round(acc_of(gf), 4), "gsm8k_std": 0.0,
                 "math_mean": round(acc_of(mf), 4), "math_std": 0.0})
# multi-seed conditions
for cond in ["teacher_1", "teacher_3"]:
    g_m, g_s, n = fmt(seed_accs(cond, "gsm8k"))
    m_m, m_s, _ = fmt(seed_accs(cond, "math"))
    rows.append({"condition": cond, "n_seeds": n,
                 "gsm8k_mean": g_m, "gsm8k_std": g_s,
                 "math_mean": m_m, "math_std": m_s})

# order for display
order = ["zero_shot", "gold", "teacher_1", "teacher_3", "teacher_8B"]
df_seeds = pd.DataFrame(rows).set_index("condition").loc[order].reset_index()
print("── Accuracy: mean ± std across seeds (GSM8K & MATH) ──")
print(df_seeds.to_string(index=False))

# H3 verdict with seed variance
t1 = next(r for r in rows if r["condition"] == "teacher_1")
t3 = next(r for r in rows if r["condition"] == "teacher_3")
print(f"\nH3 (GSM8K): teacher_1 = {t1['gsm8k_mean']:.4f} ± {t1['gsm8k_std']:.4f}  vs  "
      f"teacher_3 = {t3['gsm8k_mean']:.4f} ± {t3['gsm8k_std']:.4f}")
gap = t1["gsm8k_mean"] - t3["gsm8k_mean"]
pooled = (t1["gsm8k_std"] + t3["gsm8k_std"]) / 2 + 1e-9
print(f"  difference = {gap:+.4f}  (~{gap/pooled:.1f}x pooled std)")
print("  -> teacher_1 clearly above teacher_3 across seeds => H3 genuinely reversed.")